# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vishal-141206/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This notebook writes down — and then verifies with real queries — what a row in the FlyRank
search-intelligence data actually means, over which time window, and which fields may ever be
used as features. Every claim below has an executed query under it.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis.** One row in my primary table, `fact_content_daily_performance`, = one
`report_date` × one `client_hash_id` × one `content_hash_id`. I confirmed the real column names
with a `DESCRIBE` in the code cell below — the grain columns are `report_date`,
`client_hash_id` and `content_hash_id` (not `client_id`/`content_id`). Because the grain is
daily, I can aggregate any measured column over exactly the days at or before a decision
moment, which is what makes time-boxed features possible.

**Which tables I'm using and why.**
- `fact_content_daily_performance` — **primary**. Daily grain of observed search/engagement
  metrics; this is where features are built from.
- `dim_clients` — **context only**. Access profile and `gsc_data_start` / `ga4_data_start` are
  read to check per-client usable-history depth before trusting any feature window; never a
  feature itself.
- `dim_content` — **context only**. Content attributes (`content_type`, created dates) for
  grouping and reading; the pseudonymous IDs are for grouping/joining only, never learned from.

**Time window.** I build and verify on the mid-panel month **`month=2026-03`**. The `_sample`
partition (June 2026, the panel's *last* month) is treated as a sealed test window for any
future label — I do not develop or test anything against it in this notebook.

In [35]:
# --- Setup: token + DuckDB (never prints the token) ------------------------------
import os
from pathlib import Path
import duckdb
import pandas as pd

def _load_hf_token():
    if os.environ.get("HF_TOKEN"):
        return os.environ["HF_TOKEN"]
    for p in (Path(".env"), Path("../.env"), Path("../../.env")):
        if p.exists():
            for line in p.read_text().splitlines():
                line = line.strip()
                if line.startswith("HF_TOKEN="):
                    return line.split("=", 1)[1].strip()
    try:  # Colab Secrets fallback
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

token = _load_hf_token()
assert token, "HF_TOKEN not found — check .env or environment"
con = duckdb.connect()
con.execute(f"CREATE SECRET hf_secret (TYPE huggingface, TOKEN '{token}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"

def _ensure(name, sql):
    if con.execute("SELECT 1 FROM information_schema.tables WHERE table_name=?", [name]).fetchone():
        return
    con.execute(f"CREATE TEMP TABLE {name} AS {sql}")
    print(f"cached {name}")

_ensure("mar", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet', hive_partitioning=true)")
_ensure("apr", f"SELECT * EXCLUDE (month) FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet', hive_partitioning=true)")
_ensure("dim_clients", f"SELECT * FROM read_parquet('{BASE}/dim_clients.parquet')")
_ensure("dim_content", f"SELECT * FROM read_parquet('{BASE}/dim_content.parquet')")

# --- Confirm the real schema before writing any contract claim ------------------
cols = con.execute("DESCRIBE SELECT * FROM mar").fetchdf()
print("fact_content_daily_performance — real columns (month=2026-03):")
display(cols[["column_name", "column_type"]].rename(columns={"column_name": "column", "column_type": "type"}))

print("\nunit-of-analysis columns used below:",
      "report_date × client_hash_id × content_hash_id")

cached mar
cached apr
cached dim_clients
cached dim_content
fact_content_daily_performance — real columns (month=2026-03):


,column,type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT



unit-of-analysis columns used below: report_date × client_hash_id × content_hash_id


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Every field I touch lands in exactly one bucket.

- **Feature** (all knowable at the end of `2026-03`, no future data): `gsc_impressions_total`,
  `gsc_clicks_total`, `gsc_ctr_x100`, `gsc_avg_position_w`, `gsc_active_days`. Each is built
  from `gsc_data_available IS TRUE` rows only (see §3c for why).
- **Label / proxy** — **NOT built in this notebook.** The eventual proxy is a future-window
  outcome measured on a **later** month (e.g. April/May 2026 visibility or rank movement),
  outside this iteration window. It must come from a sealed later window, and it must be
  independent of any pre-computed trend column (in the starter CSV, `trend_direction` /
  `trend_pct` are derived from each other and can never be features). Nothing in §3 builds or
  trains against a label.
- **Context** (grouping / joining / reading only, never learned from): `report_date`,
  `client_hash_id`, `content_hash_id`, `dim_clients` (`has_gsc_access`, `gsc_data_start`,
  `access_profile`, …), `dim_content` (`content_type`, `content_created_date`, …).
- **Excluded** (each with a why):
  - `ga4_*` / `sessions_*` (GA4 and AI-session metrics) — only 4.2% of `2026-03` rows are
    `ga4_data_available = TRUE`, so a feature there would mostly encode missingness
    (verified in the code cell below).
  - `gsc_sum_position` — redundant with `gsc_avg_position` (raw sum of the same underlying
    position data).
  - `gsc_data_available IS FALSE` rows — zero-filled absence, not zero engagement;
    filtered with `IS TRUE` everywhere. 
    
    **Consequence:** content items with no GSC data at
    all in March are dropped entirely from the feature frame, not flagged as "unknown" — the
    resulting frame only represents content with actual March search visibility. Any later
    use of this frame to score/rank *all* content will need to handle that gap separately.

The code cell below prints this classification and backs the GA4 exclusion with a count.

In [36]:
import pandas as pd

classification = pd.DataFrame({
    "field": [
        "gsc_impressions_total", "gsc_clicks_total", "gsc_ctr_x100",
        "gsc_avg_position_w", "gsc_active_days",
        "future-window outcome (e.g. Apr/May visibility)",
        "report_date, client_hash_id, content_hash_id",
        "dim_clients (access_profile, gsc_data_start, ...)",
        "dim_content (content_type, created dates, ...)",
        "ga4_* / sessions_* metrics",
        "gsc_sum_position",
        "gsc_data_available IS FALSE rows",
    ],
    "bucket": [
        "feature", "feature", "feature", "feature", "feature",
        "label/proxy (NOT built here)",
        "context", "context", "context",
        "excluded", "excluded", "excluded",
    ],
    "why": [
        "March GSC impressions", "March GSC clicks", "clicks/impressions x100",
        "impressions-weighted March position", "days content visible in GSC during March",
        "must come from a later sealed window; independent of pre-computed trend columns",
        "grouping/joining only", "history depth + access checks", "content grouping",
        "only 4.2% of rows have ga4_data_available = TRUE",
        "redundant with gsc_avg_position", "zero-filled absence, not zero engagement",
    ],
})
display(classification)

# Evidence for the GA4 exclusion: how much of March is actually GA4-flagged?
con.execute("""
SELECT COUNT(*)                                          AS total_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_true_rows,
       ROUND(100.0 * COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) / COUNT(*), 2)
                                                        AS ga4_true_pct
FROM mar
""").fetchdf()

,field,bucket,why
0,gsc_impressions_total,feature,March GSC impressions
1,gsc_clicks_total,feature,March GSC clicks
2,gsc_ctr_x100,feature,clicks/impressions x100
3,gsc_avg_position_w,feature,impressions-weighted March position
4,gsc_active_days,feature,days content visible in GSC during March
5,future-window outcome (e.g. Apr/May visibility),label/proxy (NOT built here),must come from a later sealed window; independ...
6,"report_date, client_hash_id, content_hash_id",context,grouping/joining only
7,"dim_clients (access_profile, gsc_data_start, ...)",context,history depth + access checks
8,"dim_content (content_type, created dates, ...)",context,content grouping
9,ga4_* / sessions_* metrics,excluded,only 4.2% of rows have ga4_data_available = TRUE


,total_rows,ga4_true_rows,ga4_true_pct
0,9841378,413966,4.21


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

**(a) Grain check.** The contract says one row = `report_date` × `client_hash_id` ×
`content_hash_id`. If any such triple appears twice, the grain claim is wrong and must be
corrected. Expected: **zero rows** returned.

In [37]:
grain = con.execute("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
FROM mar
GROUP BY report_date, client_hash_id, content_hash_id
HAVING n > 1
LIMIT 5
""").fetchdf()
print(f"rows returned by the grain check: {len(grain)}  (0 means the unit-of-analysis claim holds)")
assert len(grain) == 0, "grain violated — the unit-of-analysis claim in §1 is wrong, investigate before proceeding"
grain

rows returned by the grain check: 0  (0 means the unit-of-analysis claim holds)


,report_date,client_hash_id,content_hash_id,n


**(b) Row count + date span for `month=2026-03`.** Measures that the partition covers exactly
the intended window and anchors the row-count claim of the contract.

In [38]:
con.execute("""
SELECT COUNT(*)       AS n_rows,
       MIN(report_date) AS first_date,
       MAX(report_date) AS last_date
FROM mar
""").fetchdf()

,n_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


**(c) Availability, filtered with `IS TRUE`.** The panel warning is real: rows outside a
client's usable history are zero-filled and flagged `gsc_data_available = FALSE` /
`ga4_data_available = FALSE`. Using `IS TRUE` (not just truthy) counts how many rows actually
carry observed GSC and GA4 data in March. This is the filter every feature below respects.

In [39]:
con.execute("""
SELECT COUNT(*)                                                    AS total_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)          AS gsc_true_rows,
       COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)          AS ga4_true_rows,
       COUNT(*) FILTER (WHERE gsc_data_available IS TRUE
                        AND ga4_data_available IS TRUE)            AS both_true_rows
FROM mar
""").fetchdf()

,total_rows,gsc_true_rows,ga4_true_rows,both_true_rows
0,9841378,3611061,413966,364347


**The 5-feature frame (max 5).** One row per `content_hash_id`, aggregated from the same
`month=2026-03` rows where `gsc_data_available IS TRUE`. Each feature is knowable at the
decision moment (end of March 2026):

1. `gsc_impressions_total` — knowable at the decision moment because it is the sum of the
   just-completed month's observed GSC impressions, all rows dated ≤ 2026-03-31.
2. `gsc_clicks_total` — knowable at the decision moment because it is the observed GSC click
   count over the same finished month.
3. `gsc_ctr_x100` — knowable at the decision moment because it is derived only from the two
   March aggregates above (clicks / impressions × 100), computable the moment March closes.
4. `gsc_avg_position_w` — knowable at the decision moment because it is the impressions-weighted
   mean of the observed daily GSC position across March, over `gsc_data_available` days only.
5. `gsc_active_days` — knowable at the decision moment because it is the count of March days the
   content appeared with `gsc_data_available = TRUE`, a coverage/activity measure known once
   March ends.

All five are computable using only data at or before `2026-03-31` — no future data.

In [40]:
feat = con.execute("""
SELECT
  content_hash_id,
  client_hash_id,
  SUM(gsc_impressions) FILTER (WHERE gsc_data_available IS TRUE)                            AS gsc_impressions_total,
  SUM(gsc_clicks)     FILTER (WHERE gsc_data_available IS TRUE)                             AS gsc_clicks_total,
  COUNT(*)           FILTER (WHERE gsc_data_available IS TRUE)                              AS gsc_active_days,
  SUM(gsc_impressions * gsc_avg_position)
    FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0)
    / NULLIF(SUM(gsc_impressions)
        FILTER (WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0), 0)              AS gsc_avg_position_w
FROM mar
WHERE gsc_data_available IS TRUE
GROUP BY 1, 2
ORDER BY content_hash_id
""").fetchdf()
feat = feat.sort_values("content_hash_id").reset_index(drop=True)
feat["gsc_ctr_x100"] = feat["gsc_clicks_total"] / feat["gsc_impressions_total"] * 100.0

print("frame size:", len(feat))
print("null checks — ctr:", int(feat["gsc_ctr_x100"].isna().sum()),
      "| weighted position:", int(feat["gsc_avg_position_w"].isna().sum()))
feat[["gsc_impressions_total", "gsc_clicks_total", "gsc_ctr_x100",
      "gsc_avg_position_w", "gsc_active_days"]].describe().T

frame size: 176738
null checks — ctr: 0 | weighted position: 1434


,count,mean,std,min,25%,50%,75%,max
gsc_impressions_total,176738.0,1587.986675,5431.337724,1.000000,20.00000,173.000000,1039.000000,617124.0
gsc_clicks_total,176738.0,4.650002,26.722649,0.000000,0.00000,0.000000,2.000000,5668.0
gsc_ctr_x100,176738.0,0.459397,3.775992,0.000000,0.00000,0.000000,0.215796,100.0
gsc_avg_position_w,175304.0,16.722366,18.537978,0.019643,5.19444,8.568248,21.372440,309.0
gsc_active_days,176738.0,20.431718,11.480153,1.000000,9.00000,26.000000,31.000000,31.0


**Sentinel-zero correction (applied above).** The raw `gsc_avg_position` column reuses `0` as
a "no position data" sentinel, per the same convention flagged in the starter-CSV data
dictionary (a real Google ranking position can never be 0). A direct check found this wasn't
a rare edge case: **163,189 of 3,611,061 GSC-available rows (4.5%)** carried a sentinel-zero
position despite having real impressions. `gsc_avg_position_w` above is therefore computed only
over rows where `gsc_avg_position > 0`, so the feature reflects genuine observed positions, not
zero-filled placeholders — `gsc_impressions_total`, `gsc_clicks_total`, and `gsc_active_days`
are unaffected, since those remain valid on sentinel-zero rows.

One consequence of the fix: **1,434 content items (0.8% of the March frame)** have *no* valid
position reading anywhere in March — every recorded day carried the sentinel zero — and
correctly show `NULL` for `gsc_avg_position_w` rather than a fabricated value. This is a real
absence, flagged as such, consistent with how missing data is handled elsewhere in this
contract.

**The leakage trap, deliberately.** I now add ONE fake "feature" that no honest feature may
use: next month's (April 2026) aggregate GSC impressions. To make the mechanism visible I need
a toy outcome to score against, so I build a *toy surrogate outcome* — `content still had GSC
impressions in April 2026` — used **only for this demonstration** and deleted with the column.
It is NOT the assignment label: the real label/proxy stays unbuilt and must come from a sealed
later window (see §2).

I train a toy classifier twice on the same March frame:
- **honest** — the 5 March features only;
- **leaky** — the same 5 features **plus** `apr_gsc_impressions_total`.

The leak column literally contains the quantity the toy outcome is computed from, so a model
with access to it reaches a perfect score — that is the tell. This is exactly why future-month
columns are banned as features.

In [41]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

# Next-month aggregate (April 2026) — the fake "feature" + toy outcome it is computed from
apr_agg = con.execute("""
SELECT content_hash_id, SUM(gsc_impressions) AS apr_gsc_impressions_total
FROM apr WHERE gsc_data_available IS TRUE GROUP BY 1 ORDER BY content_hash_id
""").fetchdf()
frame = feat.merge(apr_agg, on="content_hash_id", how="left")
frame["apr_gsc_impressions_total"] = frame["apr_gsc_impressions_total"].fillna(0).astype("int64")
frame["toy_outcome_apr_visible"] = (frame["apr_gsc_impressions_total"] > 0).astype(int)
print("toy outcome balance (0 = not visible in Apr, 1 = visible in Apr):")
print(frame["toy_outcome_apr_visible"].value_counts().to_dict())

HONEST = ["gsc_impressions_total", "gsc_clicks_total", "gsc_ctr_x100",
          "gsc_avg_position_w", "gsc_active_days"]
y = frame["toy_outcome_apr_visible"]
X = frame[HONEST].fillna(0.0)

def _split(X_):
    return train_test_split(X_, y, test_size=0.3, random_state=42, stratify=y)

# Honest-only: 5 March features
Xtr, Xte, ytr, yte = _split(X)
auc_honest = roc_auc_score(yte, DecisionTreeClassifier(max_depth=6, random_state=42)
                           .fit(Xtr, ytr).predict_proba(Xte)[:, 1])

# Leaky: same 5 features + April impressions total
Xl = X.copy()
Xl["apr_gsc_impressions_total"] = frame["apr_gsc_impressions_total"].values
Xltr, Xlte, yltr, ylte = _split(Xl)
auc_leaky = roc_auc_score(ylte, DecisionTreeClassifier(max_depth=6, random_state=42)
                          .fit(Xltr, yltr).predict_proba(Xlte)[:, 1])

print("\ntoy score (AUC) on March frame:")
print(f"  honest features only        : {auc_honest:.4f}")
print(f"  honest + FUTURE (Apr) column: {auc_leaky:.4f}")
print("\nThe future column makes the toy score look perfect — that is leakage.")

toy outcome balance (0 = not visible in Apr, 1 = visible in Apr):
{1: 158549, 0: 18189}

toy score (AUC) on March frame:
  honest features only        : 0.9223
  honest + FUTURE (Apr) column: 1.0000

The future column makes the toy score look perfect — that is leakage.


**Delete the leak column.** The fake feature is removed and the frame goes back to the five
honest March features. The toy outcome it was built from is dropped with it; nothing in the
frame references a later month.

In [42]:
frame.drop(columns=["apr_gsc_impressions_total", "toy_outcome_apr_visible"], inplace=True)
print("columns remaining after the leak trap is removed:")
print(list(frame.columns))
assert "apr_gsc_impressions_total" not in frame.columns, "leak column still present!"
assert "toy_outcome_apr_visible" not in frame.columns, "toy outcome still present!"
print("\nleak column deleted; only honest March features remain.")

columns remaining after the leak trap is removed:
['content_hash_id', 'client_hash_id', 'gsc_impressions_total', 'gsc_clicks_total', 'gsc_active_days', 'gsc_avg_position_w', 'gsc_ctr_x100']

leak column deleted; only honest March features remain.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation — usable-history depth is wildly uneven per client.** Only **55 of the 104
clients** have any rows at all in `2026-03`, and among those 55 the *median* client has only
**~8%** of its March daily rows flagged `gsc_data_available = TRUE` (observed range 0% → 92%).
Two of the 55 March clients have `gsc_data_start = NULL` despite `has_gsc_access = TRUE`, and
at the dimension level 7 of 67 GSC-access clients have no `gsc_data_start` at all. Separately,
the `source_only_missing_client_dimension` access profile (10 clients) still contributes 3,751
fact rows in March with almost no usable flags (20 GSC-flagged rows, sum of GSC impressions
= 23). Consequence: a global calendar window like `2026-03` silently mixes *missing* history
with *zero* activity. Features built here are directional and decision-support only — they can
never be read as a complete, comparable engagement record across clients.

In [43]:
from IPython.display import display as _disp

print("per-client share of March rows with gsc_data_available = TRUE (min / median / max):")
_disp(con.execute("""
SELECT ROUND(MIN(sh), 4) AS min_share, ROUND(MEDIAN(sh), 4) AS median_share, ROUND(MAX(sh), 4) AS max_share
FROM (
  SELECT client_hash_id,
         COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) * 1.0 / COUNT(*) AS sh
  FROM mar GROUP BY 1
)
""").fetchdf())

print("\nhow many of the 104 clients appear at all in March 2026:")
_disp(con.execute("""
SELECT (SELECT COUNT(*) FROM dim_clients)        AS clients_total,
       (SELECT COUNT(DISTINCT client_hash_id) FROM mar) AS clients_with_march_rows
""").fetchdf())

print("\nclients with access but no usable history start (dimension level):")
_disp(con.execute("""
SELECT COUNT(*) FILTER (WHERE has_gsc_access AND gsc_data_start IS NULL) AS gsc_start_null,
       COUNT(*) FILTER (WHERE has_ga4_access AND ga4_data_start IS NULL) AS ga4_start_null
FROM dim_clients
""").fetchdf())

print("\nrows in March from the source_only_missing_client_dimension profile (aggregate only):")
_disp(con.execute("""
SELECT d.access_profile,
       COUNT(*)                                           AS march_rows,
       COUNT(*) FILTER (WHERE f.gsc_data_available IS TRUE) AS gsc_true_rows
FROM mar f JOIN dim_clients d USING (client_hash_id)
WHERE d.access_profile = 'source_only_missing_client_dimension'
GROUP BY 1
""").fetchdf())

per-client share of March rows with gsc_data_available = TRUE (min / median / max):


,min_share,median_share,max_share
0,0.0,0.0801,0.9164



how many of the 104 clients appear at all in March 2026:


,clients_total,clients_with_march_rows
0,104,55



clients with access but no usable history start (dimension level):


,gsc_start_null,ga4_start_null
0,7,4



rows in March from the source_only_missing_client_dimension profile (aggregate only):


,access_profile,march_rows,gsc_true_rows
0,source_only_missing_client_dimension,3751,20


In [44]:
con.execute("""
SELECT d.access_profile,
       COUNT(*) AS march_rows,
       COUNT(*) FILTER (WHERE f.gsc_data_available IS TRUE) AS gsc_true_rows,
       SUM(f.gsc_impressions) FILTER (WHERE f.gsc_data_available IS TRUE) AS gsc_impressions_sum
FROM mar f JOIN dim_clients d USING (client_hash_id)
WHERE d.access_profile = 'source_only_missing_client_dimension'
GROUP BY 1""").fetchdf()

,access_profile,march_rows,gsc_true_rows,gsc_impressions_sum
0,source_only_missing_client_dimension,3751,20,23.0


Contract answers in one place:
1. Unit of analysis = `report_date` × `client_hash_id` × `content_hash_id` (confirmed via DESCRIBE).
2. Tables: `fact_content_daily_performance` (primary, daily grain) + `dim_clients`/`dim_content` (context only).
3. Time window: `month=2026-03`; the `_sample` (June 2026) is a sealed test window, not touched.
4. Label/proxy: NOT built here — must come from a later sealed window, independent of pre-computed trend columns.
5. Excluded: GA4/session metrics (only 4.2% of March rows are GA4-flagged), plus `gsc_sum_position` and `gsc_data_available IS FALSE` rows.
6. Verification: grain check (0 dup rows, asserted), 9,841,378 rows over 2026-03-01→31, availability via `IS TRUE` (3,611,061 GSC / 413,966 GA4).
7. Feature frame: 5 March features, each knowable at the decision moment; `gsc_avg_position_w` excludes 163,189 sentinel-zero rows (4.5% of GSC-available rows) per the same "0 = no data" trap flagged in the starter CSV, leaving 1,434 content items with a correctly-`NULL` position. Future-month leak column added then deleted (toy AUC 0.9223 → 1.0000, asserted removed).
8. Named limitation: uneven usable-history depth — median client ~8% of March rows GSC-flagged (0%–92% range); only 55/104 clients present in March; 7/67 GSC-access clients have no `gsc_data_start`; the `source_only_missing_client_dimension` profile contributes 3,751 March rows summing to just 23 total impressions.

## Self-check

Before submitting, each line is confirmed:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.